In [1]:
import json
import re
import time
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv


In [2]:
load_dotenv()

INPUT_FILE  = r"C:\Users\USER\Downloads\alterder_llama_dsk_8b_results.csv"
OUTPUT_FILE = r"C:\Users\USER\Downloads\alterllama_dsk_8b_results_with_pd.csv"

def strip_thinking(text: str) -> str:
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

# B: **Principal Diagnosis:** **text**  (label bold, value separately bold)
_PB = re.compile(
    r'\*\*(?:Final\s+|Most Supported\s+)?Principal Diagnosis:\*\*\s*\*\*([^*\n]{3,300})\*\*',
    re.IGNORECASE,
)

# A: **Principal Diagnosis: text**  (colon + text inside same bold span)
_PA = re.compile(
    r'\*\*(?:Final\s+|Most Supported\s+)?Principal Diagnosis:\s+([^*\n]{3,300})\*\*',
    re.IGNORECASE,
)

# C: **Principal Diagnosis:** plain text  (label bold, value plain — stops at * or newline)
_PC = re.compile(
    r'\*\*(?:Final\s+|Most Supported\s+)?Principal Diagnosis:\*\*\s+(?!\*\*)([^\n*]{3,300})',
    re.IGNORECASE,
)

# D: The [most supported] principal diagnosis [for this patient] is [the] **text**
_PD = re.compile(
    r'(?:The\s+)?(?:most\s+supported\s+)?principal diagnosis(?:\s+for\s+this\s+patient)?[:\s]+is[:\s]+(?:the\s+)?\*\*([^*\n]{3,200})\*\*',
    re.IGNORECASE,
)

# E: **text** **Rationale/Reasoning/Step-by-Step:**  (bold value then section header)
_PE = re.compile(
    r'(?:^|\n)\s*\*\*([^*\n]{3,200})\*\*\s+\*\*(?:Rationale|Reasoning|Step-by-Step)',
    re.IGNORECASE,
)

PATTERNS = [_PB, _PA, _PC, _PD, _PE]

def extract_with_regex(text: str):
    clean = strip_thinking(text)
    for pat in PATTERNS:
        matches = pat.findall(clean)
        if matches:
            return matches[0].strip().rstrip('*').strip()
    return None

# Self-test
_tests = [
    ('**Principal Diagnosis: Metastatic Breast Cancer**\n**Reasoning:** ...', 'Metastatic Breast Cancer'),
    ('**Principal Diagnosis: Coronary Artery Disease**\n**Step-by-Step Explanation:**', 'Coronary Artery Disease'),
    ('**Principal Diagnosis:** **Large Subcapsular Splenic Hematoma**  **Rationale:**', 'Large Subcapsular Splenic Hematoma'),
    ('**Principal Diagnosis:**   **Acute Coronary Syndrome Secondary to Chronic CAD**  **Rationale:**', 'Acute Coronary Syndrome Secondary to Chronic CAD'),
    ('**Principal Diagnosis:** Renal Cell Carcinoma  **Rationale:**', 'Renal Cell Carcinoma'),
    ('**Principal Diagnosis:** Small Bowel Obstruction Secondary to Parastomal Hernia  **Rationale:**', 'Small Bowel Obstruction Secondary to Parastomal Hernia'),
    ('The principal diagnosis for this patient is **Gastrointestinal Bleed**.', 'Gastrointestinal Bleed'),
    ('The principal diagnosis for this patient is the **splenic hematoma and ascites**', 'splenic hematoma and ascites'),
    ('The most supported principal diagnosis for this patient is **Renal Cell Carcinoma**.', 'Renal Cell Carcinoma'),
    ('The most supported principal diagnosis for this patient is:  **Heart Failure, Unspecified, Acute or Chronic**', 'Heart Failure, Unspecified, Acute or Chronic'),
    ('<think>blah blah</think>\n**Principal Diagnosis: Urosepsis**\n**Reasoning:**', 'Urosepsis'),
]
print('Self-test:')
for text, expected in _tests:
    got = extract_with_regex(text)
    status = 'OK' if (got and expected.lower() in got.lower()) else f'FAIL (got: {got!r})'
    print(f'  {status} | expected: {expected!r}')


Self-test:
  OK | expected: 'Metastatic Breast Cancer'
  OK | expected: 'Coronary Artery Disease'
  OK | expected: 'Large Subcapsular Splenic Hematoma'
  OK | expected: 'Acute Coronary Syndrome Secondary to Chronic CAD'
  OK | expected: 'Renal Cell Carcinoma'
  OK | expected: 'Small Bowel Obstruction Secondary to Parastomal Hernia'
  OK | expected: 'Gastrointestinal Bleed'
  OK | expected: 'splenic hematoma and ascites'
  OK | expected: 'Renal Cell Carcinoma'
  OK | expected: 'Heart Failure, Unspecified, Acute or Chronic'
  OK | expected: 'Urosepsis'


In [3]:
API_KEY = os.getenv('DEEPSEEK_API_KEY')
MODEL   = 'deepseek-v4-flash'

client = OpenAI(api_key=API_KEY, base_url='https://api.deepseek.com')

EXTRACT_SYSTEM = """You are a clinical NLP extraction assistant.
You will receive the FINAL portion of a model's clinical reasoning output.
Your only job: identify the model's stated principal diagnosis and return it as a short phrase.

Rules:
- Extract exactly what the model concludes, do not infer or add information.
- If the model explicitly names a diagnosis at the end, use that.
- If ambiguous between two candidates, pick the one stated last or most emphatically.
- Strip ICD codes, qualifiers like (with secondary ...), footnotes — keep only the core diagnosis name.
- If no clear principal diagnosis is stated, return UNCLEAR.

Return ONLY valid JSON (no markdown fences):
{"principal_diagnosis": "<short diagnosis phrase>"}"""


def extract_with_llm(tail: str) -> str:
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': EXTRACT_SYSTEM},
                {'role': 'user',   'content': tail},
            ],
            max_tokens=128,
            temperature=0.0,
        )
        raw = resp.choices[0].message.content
        clean = re.sub(r'```json|```', '', raw).strip()
        return json.loads(clean)['principal_diagnosis']
    except Exception as e:
        return f'ERROR: {e}'


In [4]:
df = pd.read_csv(INPUT_FILE)
print(f'Loaded {len(df)} rows from {INPUT_FILE}')

predicted_pds = []
methods       = []

for i, row in df.iterrows():
    text = str(row['final_answer'])

    pd_value = extract_with_regex(text)
    if pd_value:
        method = 'regex'
    else:
        clean    = strip_thinking(text)
        tail     = clean[-1200:]
        pd_value = extract_with_llm(tail)
        method   = 'llm'
        time.sleep(0.3)

    print(f'[{i+1:>2}/{len(df)}] HADM={row["HADM_ID"]} cond={row["condition"]:<15} [{method}] {str(pd_value)[:90]}')
    predicted_pds.append(pd_value)
    methods.append(method)

df['predicted_pd'] = predicted_pds
df['pd_method']    = methods

df.to_csv(OUTPUT_FILE, index=False)

n_regex = methods.count('regex')
n_llm   = methods.count('llm')
n_err   = sum(1 for v in predicted_pds if str(v).startswith('ERROR'))
print(f'\nDone -> {OUTPUT_FILE}')
print(f'  regex extracted : {n_regex}/{len(df)}')
print(f'  llm  extracted  : {n_llm}/{len(df)}')
print(f'  errors          : {n_err}')
print()
print(df[['HADM_ID', 'condition', 'predicted_pd', 'pd_method']].to_string(index=False))


Loaded 35 rows from C:\Users\USER\Downloads\alterder_llama_dsk_8b_results.csv
[ 1/35] HADM=197345 cond=0-shot          [regex] Metastatic Breast Cancer
[ 2/35] HADM=197345 cond=1-shot          [regex] Metastatic Breast Carcinoma, Unspecified (Not Otherwise Specified)
[ 3/35] HADM=197345 cond=counterfactual  [regex] Metastatic Breast Cancer with Omental Carcinomatosis
[ 4/35] HADM=197345 cond=negated_hx      [regex] Metastatic Breast Cancer
[ 5/35] HADM=197345 cond=negated_ruled_out [regex] Metastatic Breast Cancer with Omental Carcinomatosis
[ 6/35] HADM=197345 cond=negated_hedge   [regex] Metastatic Breast Cancer
[ 7/35] HADM=197345 cond=random_masked   [regex] Metastatic Breast Cancer
[ 8/35] HADM=176830 cond=0-shot          [regex] Gastrointestinal Bleed
[ 9/35] HADM=176830 cond=1-shot          [regex] Gastrointestinal Bleed
[10/35] HADM=176830 cond=counterfactual  [regex] Ischemic Colitis Secondary to Mesenteric Hypoperfusion
[11/35] HADM=176830 cond=negated_hx      [regex] Gastroi